In [ ]:
# -------------------------
# 1. GNN MODEL FOR DILI
# -------------------------

import torch
import torch.nn as nn
from torch_geometric.nn import NNConv, global_mean_pool
from rdkit import Chem
from rdkit.Chem import rdchem
from torch_geometric.data import Data

# --- helper: convert SMILES → PyG graph ---
def smiles_to_graph(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    # node features: atomic number + degree + formal charge
    atom_features = []
    for atom in mol.GetAtoms():
        atom_features.append([
            atom.GetAtomicNum(),
            atom.GetDegree(),
            atom.GetFormalCharge()
        ])

    # edge index + edge features (bond type)
    edge_index = []
    edge_attr = []
    for bond in mol.GetBonds():
        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()
        btype = int(bond.GetBondTypeAsDouble())
        edge_index.append([i, j])
        edge_index.append([j, i])
        edge_attr.append([btype])
        edge_attr.append([btype])

    data = Data(
        x=torch.tensor(atom_features, dtype=torch.float),
        edge_index=torch.tensor(edge_index, dtype=torch.long).t().contiguous(),
        edge_attr=torch.tensor(edge_attr, dtype=torch.float)
    )
    return data


# --- GNN Model ---
class DILI_MPNN(nn.Module):
    def __init__(self, hidden_dim=64):
        super().__init__()
        self.lin0 = nn.Linear(3, hidden_dim)

        nn_edge = nn.Sequential(
            nn.Linear(1, hidden_dim * hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim * hidden_dim, hidden_dim * hidden_dim)
        )

        self.conv1 = NNConv(hidden_dim, hidden_dim, nn_edge, aggr='mean')
        self.conv2 = NNConv(hidden_dim, hidden_dim, nn_edge, aggr='mean')

        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, data):
        x, edge_index, edge_attr = data.x, data.edge_index, data.edge_attr

        x = self.lin0(x)
        x = self.conv1(x, edge_index, edge_attr)
        x = self.conv2(x, edge_index, edge_attr)

        x = global_mean_pool(x, batch=torch.zeros(x.size(0), dtype=torch.long))
        return self.fc(x)


In [ ]:
smiles = "CC1=CC=CC=C1"   # example molecule
graph = smiles_to_graph(smiles)

print(graph.x.shape)         # node features
print(graph.edge_index.shape)  # edges


In [ ]:
model = DILI_MPNN(hidden_dim=64)
output = model(graph)

print("Predicted DILI probability:", float(output))


In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.BCELoss()

for epoch in range(20):
    optimizer.zero_grad()
    pred = model(graph)
    loss = criterion(pred, torch.tensor([[1.0]]))   # example label
    loss.backward()
    optimizer.step()
    print("Epoch", epoch, "loss:", float(loss))
